# Drives as a Production Line
Executed analysis companion. The tables below are real derived data. Read PROTOCOL.md before interpreting alarms. Run from the repository folder. The full pipeline is reproduced by run-study.ps1; raw data remains local.

In [1]:
from pathlib import Path
import pandas as pd, json
root = Path.cwd()
if not (root / 'derived').exists():
    root = root / 'outputs' / 'football-drives'
summary = json.loads((root / 'derived' / 'study_summary.json').read_text())
print(json.dumps(summary, indent=2))

{
  "case_count": 10,
  "eligible_cases": 8,
  "new_alarms": 1,
  "preexisting_alarms": 1,
  "insufficient_followup": 2,
  "baseline_flagged_cases": 0,
  "eligible_baseline_flagged": 0,
  "placebo_n": 50,
  "placebo_alarms": 7,
  "placebo_rate": 0.14,
  "placebo_wilson_95": [
    0.06950745262022862,
    0.2618645719852809
  ],
  "placebo_unique_teams": 48,
  "cusum_near_change": 0,
  "seed": 20261022
}


## Ten verified changes
Two cases have insufficient follow-up. The first-four-game outcome is compared with a prior-season baseline, not interpreted as a causal effect.

In [2]:
cases = pd.read_csv(root / 'derived' / 'case_results.csv')
print(cases[['team','season','baseline_n','followup_n','status','adjusted_delta']].to_string(index=False))

          team  season  baseline_n  followup_n                 status  adjusted_delta
      Nebraska    2022          11           9  new post-change alarm       -0.127820
 Arizona State    2022          11           9           no new alarm       -0.139688
  Georgia Tech    2022          11           8           no new alarm       -0.103834
      Colorado    2022          11           7           no new alarm       -0.128898
     Wisconsin    2022          12           7           no new alarm        0.134878
     Charlotte    2022          11           4      preexisting alarm        0.129493
        Auburn    2022          11           4           no new alarm       -0.013835
Michigan State    2023          12          10           no new alarm       -0.218092
     Texas A&M    2023          11           1 insufficient follow-up        0.165014
   Boise State    2023          12           3 insufficient follow-up       -0.003945


## Matched placebo dates
The same whole-season scan and new-alarm rule are used. These are not proven null cases, and the 50 units contain 48 distinct teams.

In [3]:
placebos = pd.read_csv(root / 'derived' / 'placebo_results.csv')
print(f'Placebo new alarms: {placebos.new_alarm.sum()} / {len(placebos)}')
print(placebos[['team','season','fake_first_post_date','new_alarm']].head(10).to_string(index=False))

Placebo new alarms: 7 / 50
             team  season fake_first_post_date  new_alarm
          Buffalo    2022           2022-09-17      False
          Wyoming    2022           2022-09-16      False
       Cincinnati    2022           2022-09-24      False
   Michigan State    2022           2022-10-01      False
 Eastern Michigan    2022           2022-10-01      False
              SMU    2022           2022-10-29      False
         Oklahoma    2022           2022-11-05      False
          Rutgers    2023           2023-09-09       True
             Iowa    2022           2022-10-01       True
Mississippi State    2022           2022-09-17      False


## Sensitivity
Do not select the variant that gives the most interesting result.

In [4]:
s = pd.read_csv(root / 'derived' / 'sensitivity.csv')
print(s[s.main_eligible].groupby('variant').agg(cases=('case_id','size'),new_alarms=('new_alarm','sum')).to_string())

           cases  new_alarms
variant                     
cusum_h4       8           1
cusum_h6       8           1
min40          8           2
raw            8           1
sample_sd      8           0


## Secondary tempo model
This regression is descriptive and uses game-clustered standard errors.

In [5]:
print((root / 'derived' / 'tempo_model.json').read_text())

{
  "n_team_games": 4294,
  "n_games": 2166,
  "tempo_coefficient": -0.03870342436307936,
  "ci_low": -0.12671065997529596,
  "ci_high": 0.04930381124913724,
  "clustered_se": 0.04490247591609155,
  "r_squared": 0.3785588089393792,
  "interpretation": "Descriptive association in points per drive (TD=6; FG=3; no conversions) per additional eligible snap per possession minute. Controls: offense, defense, season, field position and home field. Clustered by game. Not a causal tempo effect."
}


## Data quality
All expected games are represented; that does not prove within-game completeness.

![Effect-size intervals](charts/effect_sizes.png)

![Tempo scatter](charts/tempo_scatter.png)

In [6]:
print((root / 'derived' / 'data_audit.json').read_text())

{
  "raw_plays": 634546,
  "seasons": [
    {
      "season": 2021,
      "raw_plays": 150770
    },
    {
      "season": 2022,
      "raw_plays": 241277
    },
    {
      "season": 2023,
      "raw_plays": 242499
    }
  ],
  "eligible_plays": 285315,
  "ppa_measured": 284804,
  "team_games": 4432,
  "expected_completed_fbs_games": 2216,
  "observed_fbs_games": 2216,
  "missing_fbs_games": 0,
  "unjoined_raw_plays": 0,
  "drive_audit": {
    "eligible_snap_drives_after_ambiguity_filter": 51109,
    "ambiguous_drive_ids_excluded": 78,
    "offense_mismatch_count": 53,
    "missing_drive_records": 0,
    "garbage_drive_count": 323,
    "unknown_result_counts": {
      "Uncategorized": 242,
      "DOWNS TD": 14,
      "END OF HALF TD": 6,
      "FG TD": 1,
      "KICKOFF": 1
    },
    "valid_tempo_drives": 48692
  },
  "clean_interval_count": 3937
}


## Rebuild
From a terminal with requirements.txt installed: `python build_tables.py`, `python study.py`, `python secondary.py`, `python make_deliverables.py`. Download first with `python pipeline.py pull --seasons 2021 2022 2023` and `python pipeline.py drives --seasons 2021 2022 2023`. Set CFBD_API_KEY in the local .env file. Never commit raw/ or .env.